In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("=========================================")
print("  E-COMMERCE SHAP EXPLAINABILITY")
print("=========================================\n")

# Ensure output directories exist
os.makedirs('../../backend/models', exist_ok=True)
os.makedirs('../../backend/artifacts', exist_ok=True)

# 1. Load the Data
train_df = pd.read_csv('../../data/processed/train.csv')
X_train = train_df.drop('Revenue', axis=1)

# 2. Load the Base XGBoost Model (Uncalibrated for SHAP compatibility)
print("Loading base XGBoost model for SHAP analysis...")
with open('../../backend/models/xgboost.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

# 3. Initialize SHAP TreeExplainer
print("Initializing SHAP TreeExplainer...")
explainer = shap.TreeExplainer(xgb_model)
# NOTE: We intentionally DO NOT pickle this explainer. 
# It will be instantiated at runtime in FastAPI to prevent environment/versioning crashes.

# 4. Calculate SHAP Values (Sampled for performance)
print("Calculating SHAP values (using a sample of 2000 rows)...")
X_sample = X_train.sample(n=min(2000, len(X_train)), random_state=42)
shap_values = explainer(X_sample)

# Extract raw values array based on SHAP version handling
shap_values_array = shap_values.values if hasattr(shap_values, 'values') else shap_values

# 5. Extract and Save Top Features as CSV/JSON for Next.js Dashboard
print("Saving structured Feature Importance (CSV & JSON)...")
feature_importance = pd.DataFrame({
    "Feature": X_sample.columns,
    "MeanAbsSHAP": np.abs(shap_values_array).mean(axis=0)
}).sort_values(by="MeanAbsSHAP", ascending=False)

feature_importance.to_csv('../../backend/artifacts/shap_top_features.csv', index=False)
feature_importance.head(20).to_json('../../backend/artifacts/shap_top_features.json', orient='records')

# 6. Generate SHAP Summary Plot (Beeswarm)
print("Generating Global Summary Plot...")
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, show=False, plot_size=(10, 8))
plt.title("SHAP Summary Plot (Global Feature Impact)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../../backend/artifacts/shap_summary_plot.png', bbox_inches='tight', dpi=300)
plt.close()

# 7. Generate SHAP Bar Plot (Mean Absolute Impact)
print("Generating Mean Absolute Impact Bar Plot...")
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False, plot_size=(10, 8), color="steelblue")
plt.title("SHAP Feature Importance (Mean Absolute Impact)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../../backend/artifacts/shap_bar_plot.png', bbox_inches='tight', dpi=300)
plt.close()

# 8. Generate a Local Explainability Example (Waterfall)
print("Generating Example Waterfall Plot...")
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[0], show=False)
plt.title("Example Local Waterfall Plot (Single Customer Session)", fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig('../../backend/artifacts/shap_waterfall_example.png', bbox_inches='tight', dpi=300)
plt.close()

# 9. Generate SHAP Dependence Plot Dynamically
top_feature = feature_importance.iloc[0]["Feature"]
print(f"Generating Dependence Plot for dynamically identified top feature: {top_feature}...")

plt.figure(figsize=(10, 6))
# Dropped the 'ax' parameter to ensure compatibility across all SHAP versions
shap.dependence_plot(top_feature, shap_values_array, X_sample, show=False)
plt.title(f"SHAP Dependence Plot: {top_feature} vs. Purchase Intent", fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig(f'../../backend/artifacts/shap_dependence_{top_feature.lower()}.png', bbox_inches='tight', dpi=300)
plt.close()

print("\n=========================================")
print("SHAP ANALYSIS COMPLETE")
print("=========================================")